In [3]:
!pip install gradio

   ---------------------------------------- 0.0/24.2 MB ? eta -:--:--
   --- ------------------------------------ 2.1/24.2 MB 13.1 MB/s eta 0:00:02
   -------- ------------------------------- 5.2/24.2 MB 13.9 MB/s eta 0:00:02
   ------------- -------------------------- 7.9/24.2 MB 13.5 MB/s eta 0:00:02
   ----------------- ---------------------- 10.7/24.2 MB 13.4 MB/s eta 0:00:01
   ---------------------- ----------------- 13.6/24.2 MB 13.6 MB/s eta 0:00:01
   --------------------------- ------------ 16.5/24.2 MB 13.3 MB/s eta 0:00:01
   ------------------------------- -------- 18.9/24.2 MB 13.1 MB/s eta 0:00:01
   ------------------------------------ --- 21.8/24.2 MB 13.0 MB/s eta 0:00:01
   ---------------------------------------  24.1/24.2 MB 13.0 MB/s eta 0:00:01
   ---------------------------------------- 24.2/24.2 MB 12.3 MB/s eta 0:00:00



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [5]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API key exists and its starts with {openai_api_key[:8]}")
else:
    print("OpenAI API key not exists.")



OpenAI API key exists and its starts with sk-proj-


In [7]:
openai=OpenAI()
system_message="You are helpful Assistsnt"
def message_gpt(prompt):
    messages=[{"role":"system","content":system_message},{"role":"user","content":prompt}]
    response=openai.chat.completions.create(model="gpt-4.1-mini",messages=messages)
    return response.choices[0].message.content
              

In [8]:
message_gpt("What is Today date?")

"Today's date is April 27, 2024."

In [19]:
def shout(text):
    print("Upper case of entered text:"+ text)
    return text.upper()

In [13]:
shout('jesus')

Upper case of entered text:jesus


'JESUS'

In [15]:
gr.Interface(fn=shout,inputs="textbox",outputs="textbox",flagging_mode="never").launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Upper case of entered text:hello


In [17]:
gr.Interface(fn=shout,inputs="textbox",outputs="textbox",flagging_mode="never").launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


Upper case of entered text:hello


#Adding authentication

In [23]:
gr.Interface(fn=shout,inputs="textbox",outputs="textbox",flagging_mode="never").launch(inbrowser=True,auth=("santhu","rules"))

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


Upper case of entered text:ghghg


In [25]:
# Define this variable and then pass js=force_dark_mode when creating the Interface

force_dark_mode = """
function refresh() {
    const url = new URL(window.location);
    if (url.searchParams.get('__theme') !== 'dark') {
        url.searchParams.set('__theme', 'dark');
        window.location.href = url.href;
    }
}
"""
gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never", js=force_dark_mode).launch(inbrowser=True)

C:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\gradio\interface.py:171: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: js. Please pass these parameters to launch() instead.
  super().__init__(


* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


In [28]:
message_input=gr.Textbox(label="Your message",info="Enter your message to shout",lines=7)
message_output=gr.Textbox(label="Response:",lines=8)
view=gr.Interface(fn=shout,inputs=[message_input],outputs=[message_output],title="Shout",examples=["hello","Howdy"],flagging_mode="never")
view.launch()


* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


In [32]:
view=gr.Interface(fn=message_gpt,inputs=[message_input],outputs=[message_output],title="Message to gpt-4.1-mini",flagging_mode="never",examples=["Hello"])
view.launch()

* Running on local URL:  http://127.0.0.1:7870
* To create a public link, set `share=True` in `launch()`.


In [35]:
system_message = "You are a helpful assistant that responds in markdown without code blocks"
message_input=gr.Textbox(label="Your message",info="Enter your message to shout",lines=7)
message_output=gr.Markdown(label="Response:")
view=gr.Interface(fn=message_gpt,inputs=[message_input],outputs=[message_output],title="Shout",examples=["hello","Howdy"],flagging_mode="never")
view.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7872
* To create a public link, set `share=True` in `launch()`.


In [36]:
#Streaming
def stream_gpt(prompt):
    system_message = "You are a helpful assistant that responds in markdown without code blocks"
    messages=[{"role":"system","content":system_message},
              {"role":"user","content":prompt}]
    stream=openai.chat.completions.create(model="gpt-4.1-mini",
                                            messages=messages,
                                            stream=True)
    result=""
    for chunk in stream:
        result+=chunk.choices[0].delta.content or ""
        yield result


    



In [37]:
system_message = "You are a helpful assistant that responds in markdown without code blocks"
message_input=gr.Textbox(label="Your message",info="Enter your message to shout",lines=7)
message_output=gr.Markdown(label="Response:")
view=gr.Interface(fn=stream_gpt,inputs=[message_input],outputs=[message_output],title="Stream GPT",examples=["hello","Howdy"],flagging_mode="never")
view.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7873
* To create a public link, set `share=True` in `launch()`.


In [52]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
def stream_ollama(prompt):
    system_message = "You are a helpful assistant that responds in markdown without code blocks"
    messages=[{"role":"system","content":system_message},
              {"role":"user","content":prompt}]
    stream=ollama.chat.completions.create(model="llama3.2",
                                            messages=messages,
                                            stream=True)
    result=""
    for chunk in stream:
        result+=chunk.choices[0].delta.content or ""
        yield result

In [53]:
message_input = gr.Textbox(label="Your message:", info="Enter a message for Claude 4.5 Sonnet", lines=7)
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_ollama,
    title="Claude", 
    inputs=[message_input], 
    outputs=[message_output], 
    examples=[
        "Explain the Transformer architecture to a layperson",
        "Explain the Transformer architecture to an aspiring AI engineer",
        ], 
    flagging_mode="never"
    )
view.launch()

* Running on local URL:  http://127.0.0.1:7882
* To create a public link, set `share=True` in `launch()`.


In [49]:
def stream_model(prompt,model):
    if model=="GPT":
        result=stream_gpt(prompt)
    elif model=="ollama":
        result=stream_ollama(prompt)
    else:
        raise ValueError("Invalid Model")
    yield from result


In [54]:
message_input = gr.Textbox(label="Your message:", info="Enter a message for the LLM", lines=7)
model_selector = gr.Dropdown(["GPT", "ollama"], label="Select model", value="GPT")
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_model,
    title="LLMs", 
    inputs=[message_input, model_selector], 
    outputs=[message_output], 
    examples=[
            ["Explain the Transformer architecture to a layperson", "GPT"],
            ["Explain the Transformer architecture to an aspiring AI engineer", "ollama"]
        ], 
    flagging_mode="never"
    )
view.launch()

* Running on local URL:  http://127.0.0.1:7883
* To create a public link, set `share=True` in `launch()`.
